# Bangkok Exact-Hour Rain Intensity Neighbor Models

This notebook trains models to answer exact-hour rain intensity questions such as:

> If it is 3 PM now, will the rain at 4 PM be no rain, light, moderate, or heavy?

Exact-hour intensity targets mean:

- `rain_intensity_next_1h`: intensity class exactly at `t+1`
- `rain_intensity_next_2h`: intensity class exactly at `t+2`
- `rain_intensity_next_3h`: intensity class exactly at `t+3`
- continuing through `t+6`

Classes are based on exact future hourly precipitation:

- `0 = no_rain`: `< 0.1 mm`
- `1 = light`: `0.1 mm` to `< 2.5 mm`
- `2 = moderate`: `2.5 mm` to `< 10 mm`
- `3 = heavy`: `>= 10 mm`

Use this notebook when you want a specific future hour and the expected rain intensity class. Use the rain-any notebook when you only need to know whether rain happens anytime within a window.


## 1. Setup

In [1]:
import json
import os
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg2
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
)

pd.set_option("display.max_columns", 180)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)


## 2. Configuration

In [2]:
DB_CONFIG = {
    "host": os.getenv("PGHOST", "localhost"),
    "port": int(os.getenv("PGPORT", "5432")),
    "dbname": os.getenv("PGDATABASE", "postgres"),
    "user": os.getenv("PGUSER", "postgres"),
    "password": os.getenv("PGPASSWORD", "Pass1234"),
}

TABLE_NAME = '"OM_BKK_DATA"'
PRECOMPUTE_TABLE_NAME = '"OM_BKK_DATA_PRECOMPUTE"'
PROJECT_ROOT = Path.cwd()
MODEL_DIR = PROJECT_ROOT / "ML_Model_V2" / "trained_models" / "om_bkk_exact_hour_intensity_neighbor_models"

HORIZONS = [1, 2, 3, 4, 5, 6]
INTENSITY_CLASS_LABELS = {
    0: "no_rain",
    1: "light",
    2: "moderate",
    3: "heavy",
}
INTENSITY_CLASS_ORDER = list(INTENSITY_CLASS_LABELS)
INTENSITY_THRESHOLDS_MM = {
    "light_min": 0.1,
    "moderate_min": 2.5,
    "heavy_min": 10.0,
}

TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
SAMPLE_ROWS = None  # Example for quick smoke tests: 300_000
RANDOM_STATE = 42


## 3. Feature Columns

In [3]:
BASELINE_FEATURE_COLUMNS = [
    "temperature_2m", "relative_humidity_2m", "pressure_msl", "surface_pressure",
    "dew_point_2m", "precipitation", "cloud_cover", "wind_speed_10m",
    "wind_direction_10m", "temperature_dew_point_spread", "pressure_msl_change_3h",
    "pressure_msl_change_6h", "precipitation_lag_1h", "precipitation_lag_2h",
    "precipitation_lag_3h", "precipitation_lag_6h", "precipitation_sum_past_3h",
    "precipitation_sum_past_6h", "precipitation_sum_past_12h", "precipitation_sum_past_24h",
    "cloud_cover_lag_1h", "cloud_cover_lag_3h", "cloud_cover_lag_6h",
    "humidity_lag_1h", "humidity_lag_3h", "humidity_lag_6h",
    "wind_speed_lag_1h", "wind_speed_lag_3h", "hour_sin", "hour_cos",
    "month_sin", "month_cos", "grid_row", "grid_column", "latitude", "longitude",
]

NEIGHBOR_FEATURE_COLUMNS = [
    "neighbor_count", "neighbor_precipitation_mean", "neighbor_precipitation_max",
    "neighbor_precipitation_sum", "neighbor_rain_count", "neighbor_rain_rate",
    "neighbor_cloud_cover_mean", "neighbor_cloud_cover_max", "neighbor_relative_humidity_mean",
    "neighbor_relative_humidity_max", "neighbor_pressure_msl_mean", "neighbor_pressure_msl_min",
    "neighbor_pressure_msl_max", "neighbor_temperature_2m_mean", "neighbor_dew_point_2m_mean",
    "neighbor_temperature_dew_point_spread_mean", "neighbor_wind_speed_10m_mean",
    "neighbor_wind_speed_10m_max", "row_minus_precipitation_mean", "row_plus_precipitation_mean",
    "column_minus_precipitation_mean", "column_plus_precipitation_mean", "row_minus_cloud_cover_mean",
    "row_plus_cloud_cover_mean", "column_minus_cloud_cover_mean", "column_plus_cloud_cover_mean",
    "neighbor_precipitation_mean_minus_center", "neighbor_cloud_cover_mean_minus_center",
    "neighbor_relative_humidity_mean_minus_center", "center_pressure_msl_minus_neighbor_mean",
]

FEATURE_COLUMNS = BASELINE_FEATURE_COLUMNS + NEIGHBOR_FEATURE_COLUMNS
FUTURE_PRECIP_COLUMNS = [f"precipitation_next_{horizon}h" for horizon in HORIZONS]
TARGET_COLUMNS = [f"rain_intensity_next_{horizon}h" for horizon in HORIZONS]

print(f"Features used: {len(FEATURE_COLUMNS)}")
print(f"Exact-hour intensity targets: {TARGET_COLUMNS}")
print(f"Class labels: {INTENSITY_CLASS_LABELS}")


Features used: 66
Exact-hour intensity targets: ['rain_intensity_next_1h', 'rain_intensity_next_2h', 'rain_intensity_next_3h', 'rain_intensity_next_4h', 'rain_intensity_next_5h', 'rain_intensity_next_6h']
Class labels: {0: 'no_rain', 1: 'light', 2: 'moderate', 3: 'heavy'}


## 4. Load Precomputed Features And Build Exact-Hour Intensity Targets


In [4]:
def connect():
    return psycopg2.connect(**DB_CONFIG)


def rain_intensity_class(precipitation_mm):
    bins = [-np.inf, INTENSITY_THRESHOLDS_MM["light_min"], INTENSITY_THRESHOLDS_MM["moderate_min"], INTENSITY_THRESHOLDS_MM["heavy_min"], np.inf]
    return pd.cut(
        precipitation_mm,
        bins=bins,
        labels=INTENSITY_CLASS_ORDER,
        right=False,
    ).astype("int8")


def read_training_data(sample_rows=None):
    select_columns = [
        "grid_number",
        "local_forecast_time AS forecast_time",
        *FEATURE_COLUMNS,
        *FUTURE_PRECIP_COLUMNS,
    ]
    select_sql = ",\n        ".join(select_columns)
    query = f'''
    SELECT
        {select_sql}
    FROM {PRECOMPUTE_TABLE_NAME}
    WHERE pressure_msl_change_6h IS NOT NULL
      AND precipitation_lag_6h IS NOT NULL
      AND precipitation_sum_past_24h IS NOT NULL
      AND cloud_cover_lag_6h IS NOT NULL
      AND humidity_lag_6h IS NOT NULL
      AND wind_speed_lag_3h IS NOT NULL
      AND precipitation_next_{max(HORIZONS)}h IS NOT NULL
      AND neighbor_count > 0
    ORDER BY local_forecast_time, grid_number
    '''
    if sample_rows:
        query = f'''
        SELECT *
        FROM ({query}) complete_rows
        ORDER BY random()
        LIMIT {int(sample_rows)}
        '''
    with connect() as conn:
        data = pd.read_sql_query(query, conn, parse_dates=["forecast_time"])

    for horizon in HORIZONS:
        data[f"rain_intensity_next_{horizon}h"] = rain_intensity_class(data[f"precipitation_next_{horizon}h"])
    return data


In [ ]:
df = read_training_data(SAMPLE_ROWS)
print(df.shape)
print(df["forecast_time"].min(), "to", df["forecast_time"].max())
display(df.head())

required_columns = sorted(set(FEATURE_COLUMNS + FUTURE_PRECIP_COLUMNS + TARGET_COLUMNS))
missing_counts = df[required_columns].isna().sum().sort_values(ascending=False)
display(missing_counts[missing_counts > 0])


C:\Users\Brandon\AppData\Local\Temp\ipykernel_35052\2127766989.py:45: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql_query(query, conn, parse_dates=["forecast_time"])


## 5. Intensity Class Balance


In [ ]:
balance_rows = []
for horizon in HORIZONS:
    target = f"rain_intensity_next_{horizon}h"
    counts = df[target].value_counts().reindex(INTENSITY_CLASS_ORDER, fill_value=0)
    for class_id, rows in counts.items():
        balance_rows.append({
            "horizon_h": horizon,
            "target_column": target,
            "class_id": int(class_id),
            "class_label": INTENSITY_CLASS_LABELS[int(class_id)],
            "rows": int(rows),
            "class_rate": float(rows / len(df)),
        })

target_balance = pd.DataFrame(balance_rows)
display(target_balance)

sns.barplot(data=target_balance, x="horizon_h", y="class_rate", hue="class_label")
plt.title("Exact-Hour Rain Intensity Class Rate By Horizon")
plt.xlabel("Forecast horizon: exact hour t+h")
plt.ylabel("Class rate")
plt.show()


## 6. Chronological Split

In [ ]:
def add_time_split(data, train_fraction=0.70, validation_fraction=0.15):
    unique_times = np.array(sorted(data["forecast_time"].unique()))
    train_end = unique_times[int(len(unique_times) * train_fraction)]
    validation_end = unique_times[int(len(unique_times) * (train_fraction + validation_fraction))]
    out = data.copy()
    out["split"] = "test"
    out.loc[out["forecast_time"] < train_end, "split"] = "train"
    out.loc[(out["forecast_time"] >= train_end) & (out["forecast_time"] < validation_end), "split"] = "validation"
    return out, train_end, validation_end

model_df, train_end, validation_end = add_time_split(df, TRAIN_FRACTION, VALIDATION_FRACTION)
print(f"Train before: {train_end}")
print(f"Validation before: {validation_end}")
display(model_df.groupby("split").size().rename("rows").reset_index())

train_df = model_df[model_df["split"] == "train"]
validation_df = model_df[model_df["split"] == "validation"]
test_df = model_df[model_df["split"] == "test"]

x_train = train_df[FEATURE_COLUMNS].astype("float32")
x_validation = validation_df[FEATURE_COLUMNS].astype("float32")
x_test = test_df[FEATURE_COLUMNS].astype("float32")

## 7. Multiclass Evaluation Helpers


In [ ]:
def class_metric_rows(y_true, y_pred, probabilities, model_name, horizon, split):
    rows = []
    for class_id, class_label in INTENSITY_CLASS_LABELS.items():
        binary_true = (y_true == class_id).astype("int8")
        binary_pred = (y_pred == class_id).astype("int8")
        rows.append({
            "model": model_name,
            "target_type": "exact_hour_intensity",
            "feature_set": "neighbor_grid",
            "horizon_h": horizon,
            "split": split,
            "class_id": class_id,
            "class_label": class_label,
            "rows": int(len(y_true)),
            "class_rate": float(binary_true.mean()),
            "predicted_class_rate": float(binary_pred.mean()),
            "precision": float(precision_score(binary_true, binary_pred, zero_division=0)),
            "recall": float(recall_score(binary_true, binary_pred, zero_division=0)),
            "f1": float(f1_score(binary_true, binary_pred, zero_division=0)),
            "mean_predicted_probability": float(probabilities[:, class_id].mean()),
        })
    return rows


def evaluate_multiclass(y_true, probabilities, model_name, horizon, split):
    y_pred = probabilities.argmax(axis=1).astype("int8")
    base = {
        "model": model_name,
        "target_type": "exact_hour_intensity",
        "feature_set": "neighbor_grid",
        "horizon_h": horizon,
        "split": split,
        "rows": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_precision": float(precision_score(y_true, y_pred, labels=INTENSITY_CLASS_ORDER, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, labels=INTENSITY_CLASS_ORDER, average="macro", zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=INTENSITY_CLASS_ORDER, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, labels=INTENSITY_CLASS_ORDER, average="weighted", zero_division=0)),
        "log_loss": float(log_loss(y_true, probabilities, labels=INTENSITY_CLASS_ORDER)),
    }
    class_rows = class_metric_rows(y_true, y_pred, probabilities, model_name, horizon, split)
    confusion = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=INTENSITY_CLASS_ORDER),
        index=[f"actual_{INTENSITY_CLASS_LABELS[i]}" for i in INTENSITY_CLASS_ORDER],
        columns=[f"predicted_{INTENSITY_CLASS_LABELS[i]}" for i in INTENSITY_CLASS_ORDER],
    )
    confusion.insert(0, "horizon_h", horizon)
    confusion.insert(1, "split", split)
    return base, class_rows, confusion


## 8. Train Exact-Hour LightGBM Intensity Models


In [ ]:
try:
    from lightgbm import LGBMClassifier
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Install LightGBM first: %pip install lightgbm") from exc

models = {}
metric_rows = []
class_metric_rows_all = []
confusion_tables = []
feature_importance_rows = []

for horizon in HORIZONS:
    target = f"rain_intensity_next_{horizon}h"
    print(f"Training multiclass LightGBM for {target}...")

    y_train = train_df[target].astype("int8")
    y_validation = validation_df[target].astype("int8")
    y_test = test_df[target].astype("int8")

    model = LGBMClassifier(
        objective="multiclass",
        num_class=len(INTENSITY_CLASS_LABELS),
        class_weight="balanced",
        n_estimators=500,
        learning_rate=0.04,
        num_leaves=63,
        min_child_samples=80,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(
        x_train,
        y_train,
        eval_set=[(x_validation, y_validation)],
        eval_metric="multi_logloss",
    )
    models[horizon] = model

    importances = np.asarray(model.feature_importances_)
    if importances.ndim == 1:
        importances_by_feature = importances
    else:
        importances_by_feature = importances.sum(axis=1)
    for feature, importance in zip(FEATURE_COLUMNS, importances_by_feature):
        feature_importance_rows.append({
            "model": "lightgbm",
            "horizon_h": horizon,
            "feature": feature,
            "importance": float(importance),
            "is_neighbor_feature": feature in NEIGHBOR_FEATURE_COLUMNS,
        })

    for split_name, x_split, y_split in [
        ("validation", x_validation, y_validation),
        ("test", x_test, y_test),
    ]:
        probabilities = model.predict_proba(x_split)
        metrics, class_rows, confusion = evaluate_multiclass(
            y_split.to_numpy(), probabilities, "lightgbm", horizon, split_name
        )
        metric_rows.append(metrics)
        class_metric_rows_all.extend(class_rows)
        confusion_tables.append(confusion)

all_metrics = pd.DataFrame(metric_rows)
class_metrics = pd.DataFrame(class_metric_rows_all)
confusion_results = pd.concat(confusion_tables, ignore_index=True)
feature_importance = pd.DataFrame(feature_importance_rows)

display(all_metrics.sort_values(["horizon_h", "split"]))
display(class_metrics[class_metrics["split"] == "test"].sort_values(["horizon_h", "class_id"]))
display(confusion_results[confusion_results["split"] == "test"])


## 9. Results Charts


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
sns.lineplot(data=all_metrics[all_metrics["split"] == "test"], x="horizon_h", y="macro_f1", marker="o", ax=axes[0])
axes[0].set_title("Exact-Hour Intensity Test Macro F1 By Horizon")
sns.lineplot(data=all_metrics[all_metrics["split"] == "test"], x="horizon_h", y="log_loss", marker="o", ax=axes[1])
axes[1].set_title("Exact-Hour Intensity Test Log Loss By Horizon")
for ax in axes:
    ax.set_xlabel("Forecast horizon: exact hour t+h")
plt.show()

sns.catplot(
    data=class_metrics[class_metrics["split"] == "test"],
    x="horizon_h",
    y="f1",
    hue="class_label",
    kind="bar",
    height=5,
    aspect=1.8,
)
plt.title("Exact-Hour Test F1 By Intensity Class")
plt.xlabel("Forecast horizon: exact hour t+h")
plt.ylabel("F1")
plt.show()

display(feature_importance.sort_values(["horizon_h", "importance"], ascending=[True, False]).groupby("horizon_h").head(15))


## 10. Save Exact-Hour Intensity Models And Results


In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

for horizon, model in models.items():
    path = MODEL_DIR / f"om_bkk_rain_intensity_exact_next_{horizon}h_neighbor_grid_lightgbm.joblib"
    joblib.dump(model, path)

target_balance.to_csv(MODEL_DIR / "exact_hour_intensity_target_balance.csv", index=False)
all_metrics.to_csv(MODEL_DIR / "exact_hour_intensity_metrics.csv", index=False)
class_metrics.to_csv(MODEL_DIR / "exact_hour_intensity_class_metrics.csv", index=False)
confusion_results.to_csv(MODEL_DIR / "exact_hour_intensity_confusion_matrices.csv", index=False)
feature_importance.to_csv(MODEL_DIR / "exact_hour_intensity_lightgbm_feature_importance.csv", index=False)

metadata = {
    "table": TABLE_NAME,
    "precompute_table": PRECOMPUTE_TABLE_NAME,
    "target_type": "rain_intensity_exact_at_t_plus_h",
    "horizons": HORIZONS,
    "intensity_thresholds_mm": INTENSITY_THRESHOLDS_MM,
    "intensity_class_labels": INTENSITY_CLASS_LABELS,
    "model": "lightgbm_multiclass",
    "feature_set": "neighbor_grid",
    "feature_columns": FEATURE_COLUMNS,
    "future_precip_columns": FUTURE_PRECIP_COLUMNS,
    "target_columns": TARGET_COLUMNS,
    "train_end_exclusive": str(train_end),
    "validation_end_exclusive": str(validation_end),
    "sample_rows": SAMPLE_ROWS,
    "model_dir": str(MODEL_DIR),
    "target_balance": target_balance.to_dict(orient="records"),
    "metrics": all_metrics.to_dict(orient="records"),
    "class_metrics": class_metrics.to_dict(orient="records"),
}
(MODEL_DIR / "exact_hour_intensity_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(f"Saved exact-hour intensity models and result CSVs to {MODEL_DIR}")
